# Data Cleaning and Standardization

## Objective
Transform the raw synthetic enterprise tables into clean, standardized, analysis-ready datasets for downstream EDA, feature engineering, and machine learning.

This notebook keeps the experimentation layer visible while the reusable cleaning logic lives in `backend/app/pipelines/data_cleaning.py`.

## Architecture and Implementation Plan

1. Load the raw CSV tables created by the synthetic dataset generator.
2. Standardize column names, data types, and null handling.
3. Remove duplicates and clip impossible numeric values.
4. Persist processed tables, reports, and feature-ready outputs.
5. Build a customer feature table for future forecasting and churn modeling notebooks.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path('..').resolve()
backend_root = project_root / 'backend'
sys.path.insert(0, str(backend_root))

from app.pipelines.data_cleaning import CleaningPaths, EnterpriseDataCleaner, load_cleaned_tables

raw_dir = project_root / 'datasets'
processed_dir = project_root / 'processed'
features_dir = project_root / 'features'
reports_dir = project_root / 'reports'
paths = CleaningPaths(raw_dir=raw_dir, processed_dir=processed_dir, features_dir=features_dir, reports_dir=reports_dir)
cleaner = EnterpriseDataCleaner(paths)
table_names = [
    'customers',
    'products',
    'suppliers',
    'employees',
    'marketing_campaigns',
    'orders',
    'inventory_snapshots',
    'finance_monthly',
    'operations_daily',
    'customer_kpis',
]
table_names

: 

## Clean Raw Tables

The next cell standardizes the full dataset and writes processed CSV files plus a cleaning summary report.

In [ ]:
cleaned_tables = cleaner.clean_all(table_names)
{name: frame.shape for name, frame in cleaned_tables.items()}

## Validate Cleaning Results

Review the cleaning summary to verify row retention, duplicate removal, and null handling across the enterprise tables.

In [ ]:
summary = pd.read_csv(reports_dir / 'data_cleaning_summary.csv')
summary.sort_values('input_rows', ascending=False)

## Feature Table

Build a customer-level feature set that will feed forecasting, churn prediction, and recommendation models in later notebooks.

In [ ]:
raw_customer_tables = load_cleaned_tables(processed_dir, ['customers', 'orders'])
customer_features = cleaner.build_customer_feature_table(
    orders=raw_customer_tables['orders'],
    customers=raw_customer_tables['customers'],
)
customer_features.head()

## Summary

The enterprise tables are now standardized and ready for EDA, feature engineering, and model development. The processed layer and customer feature layer are persisted for backend use and future notebooks.